# 🚌 Bus Number Detection System
Detects bus numbers from CCTV video using YOLOv8 + Tesseract OCR.

**Run this notebook on Google Colab.**

### Before running:
1. Upload `best.pt` to this Colab session (or place it in Google Drive)
2. Set your Google Drive video File ID in the **Configuration** cell below
3. Run all cells top to bottom

## Step 1 — Install Dependencies

In [1]:
!pip install ultralytics pytesseract opencv-python-headless gdown mysql-connector-python
!apt-get install tesseract-ocr -y
!mkdir -p ./temp_images
print('✅ All dependencies installed')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 23.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 34.5/34.5 MB 60.3 MB/s eta 0:00:00
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
tesseract-ocr is already the newest version (4.1.1-2.1build1).
0 upgraded, 0 newly installed, 0 to remove and 2 not upgraded.
✅ All dependencies installed


## Step 2 — Configuration (Edit This Cell)

In [2]:
# ============================================================
#  CONFIGURATION — Change these values before running
# ============================================================

# Your Google Drive video File ID
# Get it from: drive.google.com → right-click video → Share → copy link
# The ID is the long string in the URL between /d/ and /view
GOOGLE_DRIVE_FILE_ID = '1x2rRipGAfj-GzgzHjMHHfKok4lSzwDQU'

# Model path — upload best.pt to Colab or set path inside Drive
MODEL_PATH = 'best.pt'

# Tesseract path (do NOT change this on Colab)
TESSERACT_PATH = '/usr/bin/tesseract'

# Temp folder to save cropped bus number images
IMAGES_DIR = './temp_images/'

# Detection threshold
THRESHOLD = 0.5

# Line coordinates — the virtual line a bus must cross to trigger detection
# Adjust based on your video resolution
LINE_COORDINATES = [(200, 800), (1800, 900)]

# Bus numbers to watch for (update with your bus route numbers)
VALID_BUS_NUMBERS = [9, 19, 15, 6, 5, 13, 7, 14]

# Map bus number to license plate
LICENCE_PLATE_MAP = {
    19: 'TN 84 C35619',
     9: 'TN 84 A55709',
    15: 'TN 84 C75915',
     6: 'TN 84 C85806',
     5: 'TN 84 C35805',
    13: 'TN 84 C35913',
     7: 'TN 84 C25697',
    14: 'TN 84 C15514',
}

# MySQL Database settings (leave empty strings if not using a DB)
USE_DATABASE = True   # Set to True if you have a MySQL server
DB_HOST     = 'mysql.railway.internal'
DB_PORT     = 3306
DB_USER     = 'root'
DB_PASSWORD = 'qIdGWZjhgzZJsKoVlZeODtvlwepNXDAZ'
DB_NAME     = 'railway'

print('✅ Configuration loaded')

✅ Configuration loaded


## Step 3 — Download Video from Google Drive

In [3]:
import gdown
import os

VIDEO_PATH = 'input_video.mp4'

if GOOGLE_DRIVE_FILE_ID == 'PASTE_YOUR_FILE_ID_HERE':
    print('⚠️  Please set your GOOGLE_DRIVE_FILE_ID in the Configuration cell above!')
else:
    print('📥 Downloading video from Google Drive...')
    url = f'https://drive.google.com/uc?id={GOOGLE_DRIVE_FILE_ID}'
    gdown.download(url, VIDEO_PATH, quiet=False)
    size_mb = os.path.getsize(VIDEO_PATH) / (1024 * 1024)
    print(f'✅ Video downloaded: {VIDEO_PATH} ({size_mb:.1f} MB)')

📥 Downloading video from Google Drive...


Downloading...
From: https://drive.google.com/uc?id=1x2rRipGAfj-GzgzHjMHHfKok4lSzwDQU
To: /content/input_video.mp4
100%|██████████| 100M/100M [00:01<00:00, 63.0MB/s] 

✅ Video downloaded: input_video.mp4 (95.5 MB)


## Step 4 — Upload Model (if not already uploaded)

In [4]:
import os

if not os.path.exists(MODEL_PATH):
    print('best.pt not found. Uploading now...')
    from google.colab import files
    uploaded = files.upload()  # Upload best.pt here
    print('✅ Model uploaded:', list(uploaded.keys()))
else:
    size_mb = os.path.getsize(MODEL_PATH) / (1024 * 1024)
    print(f'✅ Model found: {MODEL_PATH} ({size_mb:.1f} MB)')

best.pt not found. Uploading now...


✅ Model uploaded: []


## Step 5 — Setup Database Connection (Optional)

In [5]:
db_connection = None
cursor = None

if USE_DATABASE:
    import mysql.connector
    try:
        db_connection = mysql.connector.connect(
            host=DB_HOST,
            user=DB_USER,
            password=DB_PASSWORD,
            database=DB_NAME
        )
        cursor = db_connection.cursor()
        print('✅ Database connected successfully')
    except Exception as e:
        print(f'❌ Database connection failed: {e}')
        print('   Detections will be printed to console only.')
else:
    print('ℹ️  Running without database. Results will print to console.')

❌ Database connection failed: 2003: Can't connect to MySQL server on 'mysql.railway.internal:3306' (Errno -2: Name or service not known)
   Detections will be printed to console only.


## Step 6 — Run Bus Detection

In [6]:
import os
import cv2
import datetime
from ultralytics import YOLO
from pytesseract import pytesseract
from collections import Counter
from statistics import mode

# ── Setup ──────────────────────────────────────────────────
pytesseract.tesseract_cmd = TESSERACT_PATH
os.makedirs(IMAGES_DIR, exist_ok=True)

cap = cv2.VideoCapture(VIDEO_PATH)
if not cap.isOpened():
    raise FileNotFoundError(f'Cannot open video: {VIDEO_PATH}')

ret, frame = cap.read()
H, W, _ = frame.shape
print(f'📹 Video loaded — Resolution: {W}x{H}')

# Output video
video_name = os.path.splitext(os.path.basename(VIDEO_PATH))[0]
video_out_path = f'{video_name}_out.mp4'
out = cv2.VideoWriter(
    video_out_path,
    cv2.VideoWriter_fourcc(*'mp4v'),
    int(cap.get(cv2.CAP_PROP_FPS)),
    (W, H)
)

model = YOLO(MODEL_PATH)
print('✅ YOLO model loaded')

# ── State ──────────────────────────────────────────────────
l = []                    # OCR results per batch
saved_image_paths = []    # Paths of cropped images
detected_in  = set()      # Buses that have come IN
detected_out = set()      # Buses that have gone OUT
last_detected = 0         # Last confirmed bus number
frame_count = 0

line_color = (0, 0, 255)

print('\n🚌 Starting detection...\n')

# ── Main Loop ──────────────────────────────────────────────
while ret:
    frame_count += 1
    results = model(frame, verbose=False)[0]

    for result in results.boxes.data.tolist():
        x1, y1, x2, y2, score, class_id = result

        if score > THRESHOLD:
            # Check if bus crossed the virtual line
            if y1 <= LINE_COORDINATES[0][1] and y2 >= LINE_COORDINATES[0][1]:
                # Draw bounding box
                cv2.rectangle(frame, (int(x1), int(y1)), (int(x2), int(y2)), (0, 255, 0), 4)
                cv2.putText(
                    frame,
                    results.names[int(class_id)].upper(),
                    (int(x1), int(y1 - 10)),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.3, (0, 255, 0), 3, cv2.LINE_AA
                )
                # Save cropped image of bus number
                if len(saved_image_paths) < 5:
                    crop = frame[int(y1):int(y2), int(x1):int(x2)]
                    img_name = os.path.join(IMAGES_DIR, f'crop_{len(saved_image_paths)}.jpg')
                    cv2.imwrite(img_name, crop)
                    saved_image_paths.append(img_name)

    # Once we have 5 cropped images, run OCR
    if len(saved_image_paths) == 5:
        for img_path in saved_image_paths:
            img = cv2.imread(img_path)
            text = pytesseract.image_to_string(
                img,
                config='-l eng --psm 9 -c tessedit_char_whitelist=1234567890'
            )
            numbers = [int(n) for n in text.split() if n.isdigit() and int(n) in VALID_BUS_NUMBERS]
            l.extend(numbers)

        saved_image_paths.clear()

        # Clean up temp images
        for f_entry in os.scandir(IMAGES_DIR):
            if f_entry.is_file():
                os.remove(f_entry.path)

        if not l:
            print(f'  [Frame {frame_count}] No valid bus number detected in this batch')
        else:
            bus_number = mode(l)
            current_time = datetime.datetime.now()
            today_date   = datetime.date.today()
            licence_plate = LICENCE_PLATE_MAP.get(bus_number, 'Unknown')

            # New bus arriving
            if bus_number not in detected_in and bus_number not in detected_out:
                print(f'  ✅ BUS IN  | Bus #{bus_number} | Plate: {licence_plate} | Time: {current_time}')
                detected_in.add(bus_number)
                last_detected = bus_number

                if USE_DATABASE and cursor:
                    sql = "UPDATE bus_number_detection SET Out_time = %s, Out_Date = %s WHERE bus_number = %s AND Out_time IS NULL"
                    cursor.execute(sql, (bus_number, current_time, today_date, licence_plate))
                    db_connection.commit()

            # Bus leaving (seen again after being tracked as IN)
            elif bus_number in detected_in and last_detected != bus_number:
                if bus_number not in detected_out:
                    print(f'  🚌 BUS OUT | Bus #{bus_number} | Time: {current_time}')
                    detected_out.add(bus_number)
                    detected_in.discard(bus_number)

                    if USE_DATABASE and cursor:
                        sql = "UPDATE bus_number_detection SET Out_time = %s, Out_Date = %s WHERE bus_number = %s AND Out_time = '00:00:00'"
                        cursor.execute(sql, (current_time, today_date, bus_number))
                        db_connection.commit()
        l.clear()

    # Draw virtual detection line
    cv2.line(frame, LINE_COORDINATES[0], LINE_COORDINATES[1], line_color, 2)

    out.write(frame)
    ret, frame = cap.read()

# ── Cleanup ────────────────────────────────────────────────
cap.release()
out.release()

if USE_DATABASE and cursor:
    cursor.close()
    db_connection.close()

print(f'\n✅ Detection complete!')
print(f'   Output video saved: {video_out_path}')
print(f'   Total frames processed: {frame_count}')

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


KeyboardInterrupt: 

## Step 7 — Download Output Video

In [ ]:
from google.colab import files
import os

if os.path.exists(video_out_path):
    print(f'📥 Downloading output video: {video_out_path}')
    files.download(video_out_path)
else:
    print('⚠️  Output video not found. Check if detection ran successfully.')